# Inspect intermediate results

:::{autolink-concat}
:::

````{margin}
```{warning}
{class}`graphviz.Source` requires your system to have DOT installed, see {doc}`Installation <graphviz:index>`.
```
````

QRules does not go from a reaction description to a {class}`.ReactionInfo` object in one step. The functions in {mod}`qrules.workflow` first create {class}`.QNProblemSet`s, solve these as sets of quantum numbers, and only match {class}`.Particle` definitions to those quantum numbers at the end. Each intermediate result can be inspected and modified, which is what this page is about. See {doc}`/usage/reaction` for the workflow as a whole and {doc}`/usage/visualize` for the rendering functions used here.

The {func}`~qrules.workflow.create_qn_problem_sets` and {func}`~qrules.workflow.find_solutions` functions provide the high-level workflow. The lower-level functions in {mod}`qrules.workflow` expose each stage separately, so intermediate results can be inspected or modified before continuing.

In [ ]:
import graphviz
from IPython.display import Markdown

import qrules
from qrules.conservation_rules import (
    parity_conservation,
    spin_magnitude_conservation,
    spin_validity,
)
from qrules.quantum_numbers import EdgeQuantumNumbers, NodeQuantumNumbers
from qrules.solving import dict_set_intersection, filter_quantum_number_problem_set
from qrules.workflow import (
    AllowedIntermediateParticles,
    InteractionConfig,
    create_qn_problem_sets,
    solve,
)

(problem-sets)=

## {class}`.QNProblemSet`s

The {func}`~qrules.workflow.create_qn_problem_sets` function generates a {class}`~qrules.workflow.QNProblemSetCollection` that can be inspected and modified before solving. Configure the allowed interaction types through an {class}`~qrules.workflow.InteractionConfig`:

In [ ]:
from qrules.particle import load_pdg
from qrules.settings import InteractionType, create_interaction_settings

particle_db = load_pdg()
interaction_config = InteractionConfig(
    create_interaction_settings(
        formalism="canonical-helicity",
        particle_db=particle_db,
    )
)
interaction_config.set_allowed_interaction_types([
    InteractionType.STRONG,
    InteractionType.EM,
])
qn_problem_sets = create_qn_problem_sets(
    initial_state=["J/psi(1S)"],
    final_state=["K0", "Sigma+", "p~"],
    formalism="canonical-helicity",
    particle_db=particle_db,
    interaction_config=interaction_config,
)

The {attr}`~qrules.workflow.QNProblemSetCollection.problem_sets` attribute is a {obj}`dict` with interaction strengths as keys and {obj}`list`s of {class}`.QNProblemSet`s as values. The collection also retains the context needed by later workflow stages.

In [ ]:
sorted(qn_problem_sets.problem_sets, reverse=True)

In [ ]:
qn_problem_set = qn_problem_sets.problem_sets[60.0][0]
dot = qrules.io.asdot(qn_problem_set, render_node=True)
graphviz.Source(dot)

In Mermaid, a {class}`.QNProblemSet` is rendered as follows:

In [ ]:
source = qrules.io.asmermaid(qn_problem_set, render_node=True, markdown=True)
Markdown(source)

## Quantum number solutions

The high-level {func}`~qrules.workflow.find_solutions` function turns the collection into a {class}`.ReactionInfo` object. To inspect the quantum-number solutions first, call the lower-level {func}`~qrules.workflow.solve` function. Its output is a {obj}`dict` with strengths as keys and lists of {class}`.QNProblemSet` and {class}`.QNResult` pairs as values:

In [ ]:
qn_solutions = solve(
    qn_problem_sets.problem_sets,
    qn_problem_sets.intermediate_particles,
)
{strength: len(values) for strength, values in qn_solutions.items()}

Each list entry is a {obj}`tuple` containing a {class}`.QNProblemSet` (compare {ref}`problem-sets`) and a {class}`.QNResult`:

In [ ]:
strong_qn_solutions = qn_solutions[3600.0]
qn_problem_set, qn_result = strong_qn_solutions[0]

In [ ]:
dot = qrules.io.asdot(qn_problem_set, render_node=True)
graphviz.Source(dot)

In [ ]:
dot = qrules.io.asdot(qn_result, render_node=True)
graphviz.Source(dot)

### Filtering quantum number problem sets

Sometimes, only a certain subset of quantum numbers and conservation rules are relevant, or the default workflow gives too many solutions for the follow-up analysis.
The {func}`.filter_quantum_number_problem_set` function can be used to produce a {class}`.QNProblemSet` where only the desired quantum numbers and conservation rules are considered when fed back to the solver.

In [ ]:
desired_edge_properties = {
    EdgeQuantumNumbers.spin_magnitude,
    EdgeQuantumNumbers.parity,
}
filtered_qn_problem_set = filter_quantum_number_problem_set(
    qn_problem_set,
    edge_rules={spin_validity},
    node_rules={spin_magnitude_conservation, parity_conservation},
    edge_properties=desired_edge_properties,
    node_properties={
        NodeQuantumNumbers.l_magnitude,
        NodeQuantumNumbers.s_magnitude,
    },
)

In [ ]:
dot = qrules.io.asdot(filtered_qn_problem_set, render_node=True)
graphviz.Source(dot)

Feed the filtered problem set back into {func}`~qrules.workflow.solve`. The intermediate-particle selection has to be reduced to the desired quantum numbers as well, because {func}`~qrules.workflow.solve` uses it to constrain the intermediate edges:

In [ ]:
filtered_particles = AllowedIntermediateParticles(
    particles=tuple(
        dict_set_intersection(properties, desired_edge_properties)
        for properties in qn_problem_sets.intermediate_particles.particles
    ),
    names=qn_problem_sets.intermediate_particles.names,
)
filtered_qn_solutions = solve(
    {3600.0: [filtered_qn_problem_set]},
    filtered_particles,
)
filtered_qn_result = filtered_qn_solutions[3600.0][0][1].solutions[6]

In [ ]:
dot = qrules.io.asdot(filtered_qn_result, render_node=True)
graphviz.Source(dot)

In Mermaid, the {class}`.QNProblemSet` is rendered as follows:

In [ ]:
source = qrules.io.asmermaid(qn_problem_set, render_node=True, markdown=True)
Markdown(source)

And analogously, the filtered {class}`.QNResult` can be visualized:

In [ ]:
source = qrules.io.asmermaid(filtered_qn_result, render_node=True, markdown=True)
Markdown(source)

:::{seealso}
[](../visualize.ipynb)
:::